In [33]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

# The URL that lists all commands alphabetically
url = "https://man7.org/linux/man-pages/dir_all_alphabetic.html"

response = requests.get(url)
if response.status_code != 200:
    print(f"Failed to retrieve page. Status code: {response.status_code}")
    exit()

soup = BeautifulSoup(response.text, "html.parser")

# 1. Find the <pre> block that contains the list of commands
pre_tag = soup.find("pre")
if not pre_tag:
    print("No <pre> block found on the page.")
    exit()

# 2. Within <pre>, each command is linked via <a href="./manX/...">
#    We'll collect command name, man-page section, and description.
commands_data = []

# Select all <a> tags that have an href ending with .html
for link in pre_tag.find_all("a", href=True):
    # Some <a> tags are just 'top' or anchor jumps (id="letter_a"), skip those
    if link.get("id"):
        continue
    
    # Example href: "./man3/a64l.3.html"
    if link["href"].startswith("./man"):
        # The link text is something like: "a64l(3)"
        cmd_text = link.get_text(strip=True)
        
        # 3. The description usually appears in the next sibling text, e.g.:
        #    " - convert between long and base-64"
        #    We need to remove the leading " - ".
        desc_node = link.next_sibling  # This might be a text node
        desc = ""
        if desc_node and isinstance(desc_node, str):
            desc = desc_node.strip()
            if desc.startswith("-"):
                desc = desc[1:].strip()  # Remove the leading '-' and extra spaces
        
        # 4. Separate the command name from the section. For example:
        #    "a64l(3)" -> command_name="a64l", command_section="3"
        match = re.match(r"^(.+)\((\d+[a-zA-Z0-9]*)\)$", cmd_text)
        if match:
            command_name = match.group(1)
            command_section = match.group(2)
        else:
            command_name = cmd_text
            command_section = ""
        
        commands_data.append((command_name, command_section, desc))

In [38]:
# Create a DataFrame with three columns.
df = pd.DataFrame(commands_data, columns=["Command", "Section", "Description"])

# Filter for shell commands (man page section 1 or 1p).
df_filtered = df[df["Section"].isin(["1", "1p", "8"])]

In [39]:
# Keep only the "Command" and "Description" columns.
df_filtered = df_filtered[["Command", "Description"]]

# Remove rows with empty descriptions
df_filtered = df_filtered[df_filtered["Description"].str.strip() != ""]

In [37]:
# df_filtered.to_csv("commands.csv", index=False)

Now we clean the commands' description

In [40]:
import nltk
from nltk.corpus import stopwords
import re

# Download the list of English stopwords (only needed once)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_description(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation using regex
    text = re.sub(r'[^\w\s]', '', text)
    # Split the text into individual words
    words = text.split()
    # Filter out stopwords
    words = [word for word in words if word not in stop_words]
    # Join the filtered words back into a string
    return " ".join(words)

# Apply the cleaning function to the "Description" column of your DataFrame
df_filtered['Description'] = df_filtered['Description'].apply(clean_description)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tferreira/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [42]:
# df_filtered.to_csv("commands_cleaned.csv", index=False)